# 02 — Wrench-rate Jacobian and sign-convention validation

The supplied appendix ends with

$$\dot{\mathbf w}_m=J_q\dot{\mathbf q}+J_\chi\dot{\boldsymbol\chi}.$$

This notebook constructs the EPM/source and IPM/receiver twist-to-wrench Jacobians by central differences and verifies the local prediction. This is the reference against which an analytic derivation should be tested.


In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.jacobians import (
    axis_rate, damped_pseudoinverse, pose_wrench_jacobians,
    rotate_axis_first_order,
)
from src.magnetics import magnetic_wrench, normalize
np.set_printoptions(precision=6, suppress=True)


## Pose, moments, and numerical Jacobians

Twists are ordered $[v_x,v_y,v_z,\omega_x,\omega_y,\omega_z]^T$. Rotations are right-handed spatial perturbations. The returned Jacobians satisfy

$$\dot w\approx J_E v_E+J_I v_I.$$


In [ ]:
p_E = np.array([0.015, 0.025, -0.040])
e_E = normalize(np.array([0.25, -0.15, 0.96]))
p_I = np.array([0.085, -0.005, 0.105])
e_I = normalize(np.array([-0.30, 0.82, 0.48]))
m_E, m_I = 2.5, 0.050

J_E, J_I = pose_wrench_jacobians(p_E, e_E, p_I, e_I, m_E, m_I)
print("source/EPM Jacobian shape:", J_E.shape)
print("receiver/IPM Jacobian shape:", J_I.shape)
print("translation cancellation norm:", np.linalg.norm(J_E[:, :3] + J_I[:, :3]))
assert np.allclose(J_E[:, :3], -J_I[:, :3], rtol=1e-8, atol=1e-10)


Translation of both magnets by the same velocity cannot change the relative geometry, so the two translation blocks must cancel. This is a powerful sign check.


In [ ]:
v_E = np.array([0.003, -0.002, 0.001, 0.06, -0.04, 0.03])
v_I = np.array([-0.001, 0.0015, 0.0005, -0.02, 0.01, 0.04])
dt = 1e-4

w0 = magnetic_wrench(p_I-p_E, m_E*e_E, m_I*e_I)
predicted = J_E @ v_E + J_I @ v_I

p_E_next = p_E + dt*v_E[:3]
p_I_next = p_I + dt*v_I[:3]
e_E_next = rotate_axis_first_order(e_E, dt*v_E[3:])
e_I_next = rotate_axis_first_order(e_I, dt*v_I[3:])
w1 = magnetic_wrench(p_I_next-p_E_next, m_E*e_E_next, m_I*e_I_next)
measured = (w1-w0)/dt

print("predicted wrench rate:", predicted)
print("finite-step rate       :", measured)
print("relative error         :", np.linalg.norm(predicted-measured)/np.linalg.norm(measured))
assert np.allclose(predicted, measured, rtol=4e-3, atol=1e-8)


## The unit-axis derivative

For a spatial angular velocity,

$$\dot{\hat e}=\omega\times\hat e=[\hat e]_\times^T\omega.$$

Its derivative must lie in the tangent plane of the unit sphere, so $\hat e^T\dot{\hat e}=0$.


In [ ]:
omega = np.array([0.4, -0.2, 0.1])
e_dot = axis_rate(e_E, omega)
print("e_dot:", e_dot)
print("e.T e_dot (must be zero):", e_E @ e_dot)
assert abs(e_E @ e_dot) < 1e-14


## Damped differential inverse

Assume the EPM end-effector twist is directly commanded. Given a desired wrench rate $y$ after compensating IPM motion, a damped solution is

$$v_E=J_E^T(J_EJ_E^T+\rho^2I)^{-1}y.$$

The numerical value of $\rho$ is meaningful only after force/torque and translation/rotation scaling. This example uses explicit row scaling solely for demonstration.


In [ ]:
# Convert torque rows to an equivalent force using a 50 mm characteristic length.
length_scale = 0.05
W = np.diag([1, 1, 1, 1/length_scale, 1/length_scale, 1/length_scale])
J_scaled = W @ J_E
desired_wrench_rate = np.array([0.003, -0.002, 0.001, 3e-5, -2e-5, 1e-5])
y_scaled = W @ desired_wrench_rate

for rho in [1e-9, 1e-4, 1e-2]:
    v_command = damped_pseudoinverse(J_scaled, rho) @ y_scaled
    residual = J_E @ v_command - desired_wrench_rate
    print(f"rho={rho:8.1e} |v|={np.linalg.norm(v_command):.5g} "
          f"scaled residual={np.linalg.norm(W@residual):.5g}")


## Implementation checklist

1. Verify the relative-position derivative before grouping terms.
2. Specify spatial versus body angular velocity.
3. Check the analytic Jacobian at many nonsymmetric poses.
4. Scale mixed force/torque and translation/rotation units.
5. Monitor rank, saturation, and residual task slack online.
6. Include internal motion, localization delay, and contact disturbances in the closed-loop test.
